In [1]:
# Импорты, версии и общие настройки — всё в одном месте, в начале блокнота
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:                       # sklearn нужен только проектам с моделями (с главы 6)
    import sklearn
    from sklearn.dummy import DummyRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
    from sklearn.model_selection import train_test_split
except ImportError:
    sklearn = None

RANDOM_STATE = 42          # одно число для всей случайности: разбиения, модели, выборки
# папка с данными: data/ рядом с блокнотом (свой проект) или ../data (блокнот в папке курса)
DATA = next((p for p in (Path('data'), Path('../data')) if p.exists()), Path('data'))

pd.set_option('display.max_columns', 30)
print('Python', sys.version.split()[0], '| данные:', DATA.resolve())
for lib in (pd, np, matplotlib, sklearn):
    if lib is None:
        continue
    print(f'{lib.__name__:<11}{lib.__version__}')
from IPython.display import Markdown, display

Python 3.12.14 | данные: /private/tmp/course-learner-2026/data
pandas     3.0.6
numpy      2.4.6
matplotlib 3.10.9
sklearn    1.8.0


# Осень сети «Зерно»

## 1. Цель
Проверить шесть описательных гипотез для руководителя. До расчёта: H1 — оба прироста >0; H2 — пересечение лидеров по местам и выручке; H3 требует эксперимента; H4 — чек с картой выше ≥5%; H5 — пик выходной до 12; H6 — разница маржи ≥3 п.п.

**Вывод:** результатом будет таблица вердиктов с ограничениями, а не обещание эффекта акции.

## 2. Данные
Три CSV за сентябрь–ноябрь 2026. Одна строка — заказ; справочники соединяем many-to-one.

In [2]:
orders = pd.read_csv(DATA / "orders_3m.csv", parse_dates=["datetime"])
shops = pd.read_csv(DATA / "shops.csv")
menu = pd.read_csv(DATA / "menu.csv")
full = orders.merge(shops, on="shop_id", how="left", validate="many_to_one").merge(menu, on="drink", how="left", validate="many_to_one")
same_rows = len(full) == len(orders)
print(full.head().to_string(), same_rows)

  order_id            datetime shop_id     drink size  qty  amount client_id   channel   city    name      opened  seats category  price_s  cost_s
0  O-00001 2026-09-01 10:31:00   PRM-1  Капучино    S    2     380     C1237       зал  Пермь  Ленина  2021-03-01     24     кофе      190      45
1  O-00002 2026-09-01 13:34:00   PRM-1       Чай    S    2     240     C1074   с собой  Пермь  Ленина  2021-03-01     24  не кофе      120      15
2  O-00003 2026-09-01 20:42:00   PRM-1       Раф    M    1     325       NaN   с собой  Пермь  Ленина  2021-03-01     24     кофе      260      70
3  O-00004 2026-09-01 19:55:00   PRM-1     Латте    M    1     262     C1153  доставка  Пермь  Ленина  2021-03-01     24     кофе      210      50
4  O-00005 2026-09-01 19:32:00   PRM-1  Капучино    M    1     238     C1048   с собой  Пермь  Ленина  2021-03-01     24     кофе      190      45 True


**Вывод:** число строк сохранилось, показаны ключи и признаки справочников; по ID без карты нельзя считать всех гостей.

## 3. Предобработка
Дату загрузили как datetime; незаполненный client_id сохраняем как заказ без карты. validate защищает от размножения строк.

**Вывод:** фильтрация анонимных заказов изменила бы бизнес-вопрос, поэтому они остаются.

## 4.1. H1 — рост сети

In [3]:
monthly = full.groupby(full.datetime.dt.to_period("M")).agg(revenue=("amount","sum"),orders=("order_id","size"),avg_check=("amount","mean"),clients=("client_id","nunique"))
monthly["avg_check"] = monthly.avg_check.round(1)
nov_vs_oct = round(monthly.revenue.pct_change().iloc[-1]*100,1)
print(monthly, nov_vs_oct)

          revenue  orders  avg_check  clients
datetime                                     
2026-09    220241     651      338.3      201
2026-10    233756     700      333.9      208
2026-11    256198     738      347.2      214 9.6


**Вывод:** оба прироста положительны. Это рост выручки сети; число активных карт не устанавливает привлечение новых людей.

## 4.2. H2 — размер кофейни

In [4]:
by_shop = full.groupby("shop_id").agg(city=("city","first"),revenue=("amount","sum"),seats=("seats","first"))
by_shop["per_seat"] = (by_shop.revenue/by_shop.seats).round()
by_shop = by_shop.sort_values("per_seat",ascending=False)
leader = by_shop.per_seat.idxmax()
leader_by_revenue = by_shop.revenue.idxmax()
print(by_shop, leader, leader_by_revenue)

           city  revenue  seats  per_seat
shop_id                                  
PRM-1     Пермь   186163     24    7757.0
KZN-1    Казань   181365     30    6046.0
KZN-2    Казань    98340     18    5463.0
TMN-1    Тюмень    96489     20    4824.0
PRM-2     Пермь    75377     16    4711.0
UFA-1       Уфа    72461     22    3294.0 PRM-1 PRM-1


**Вывод:** лидер PRM-1 имеет 24 места, крупнейшая KZN-1 — 30; H2 не подтвердилась. Выручка на место полезна для сравнения использования мест.

## 4.3. H3 — сезонный напиток

In [5]:
nov=full[full.datetime.dt.month==11]
oct=full[full.datetime.dt.month==10]
seasonal_share=round(100*nov.loc[nov.drink=="Сезонный","amount"].sum()/nov.amount.sum(),1)
check_change=round(100*(nov.amount.mean()/oct.amount.mean()-1),1)
print(seasonal_share,check_change)

15.6 4.0


**Вывод:** напиток и рост чека совпали во времени. Причину этим сравнением не установить; нужен эксперимент.

## 4.4. H4 — карта

In [6]:
avg_card = round(full.loc[full.client_id.notna(),"amount"].mean(),1)
avg_no_card = round(full.loc[full.client_id.isna(),"amount"].mean(),1)
diff_pct = round((full.loc[full.client_id.notna(),"amount"].mean()/full.loc[full.client_id.isna(),"amount"].mean()-1)*100,1)
print(avg_card, avg_no_card, diff_pct)

343.2 335.2 2.4


In [7]:
conclusion = f"**Вывод:** H4 не подтвердилась: чек с картой {avg_card} ₽, без {avg_no_card} ₽, разница {diff_pct}%, меньше 5%. Изменение после выдачи карты нужно измерить отдельно."
display(Markdown(conclusion))

**Вывод:** H4 не подтвердилась: чек с картой 343.2 ₽, без 335.2 ₽, разница 2.4%, меньше 5%. Изменение после выдачи карты нужно измерить отдельно.

**Вывод по карте:** наблюдаемая разница 2.4% ниже заранее назначенного порога 5%; H4 не подтвердилась. Причинный эффект карты этим сравнением не измерен.

## 4.5. H5 — часы пик

In [8]:
load = full.pivot_table(index=full.datetime.dt.weekday,columns=full.datetime.dt.hour,values="order_id",aggfunc="count",fill_value=0)
peak = load.stack().idxmax()
print(load, peak)

datetime  8   9   10  11  12  13  14  15  16  17  18  19  20  21
datetime                                                        
0         15  26  27  17  28  22  25  10  15  23  17  14  15   8
1         12  39  29  22  28  28  13   9  16  18  29  19  10   5
2         14  41  24  15  25  23  23   7  17  20  17  25  13  11
3         16  19  26  20  18  36  22  16  14  14  23  17  15   4
4         28  39  24  29  23  21  10   9  17  19  18  17  15   8
5         27  56  42  24  29  36  26  25  23  22  23  25  16   6
6         25  35  36  28  29  36  26  21  17  24  28  23  17  13 (np.int32(5), np.int32(9))


**Вывод:** суббота в 9 — максимум всей сети за осень, H5 подтвердилась. Это не готовая норма сотрудников одной смены.

## 4.6. H6 — маржа

In [9]:
SIZE_K = {"S":1,"M":1.25,"L":1.5}
tmp = full.assign(cost=full.cost_s*full["size"].map(SIZE_K)*full.qty)
g = tmp.groupby("category").agg(revenue=("amount","sum"),cost=("cost","sum"))
margin = ((g.revenue-g.cost)/g.revenue*100).round(1)
print(margin)

category
кофе       75.9
не кофе    77.3
dtype: float64


**Вывод:** маржа 75.9% и 77.3%, разница меньше 3 п.п., H6 не подтвердилась. Сравнение категорий не заменяет расчёт по напиткам.

## 5. Общий вывод

In [10]:
monthly=full.groupby(full.datetime.dt.to_period("M")).amount.sum()
b=full.groupby("shop_id").agg(revenue=("amount","sum"),seats=("seats","first"))
a=full.loc[full.client_id.notna(),"amount"].mean()
z=full.loc[full.client_id.isna(),"amount"].mean()
load=full.groupby([full.datetime.dt.weekday,full.datetime.dt.hour]).size()
peak=load.idxmax()
t=full.assign(cost=full.cost_s*full["size"].map({"S":1,"M":1.25,"L":1.5})*full.qty).groupby("category").agg(revenue=("amount","sum"),cost=("cost","sum"))
m=(t.revenue-t.cost)/t.revenue*100
def verdict(x):
    return "подтвердилась" if x else "не подтвердилась"
verdicts={"H1":verdict((monthly.pct_change().dropna()>0).all()),"H2":verdict(bool(set(b[b.seats==b.seats.max()].index)&set(b[b.revenue==b.revenue.max()].index))),"H4":verdict((a/z-1)*100>=5),"H5":verdict(peak[0] in (5,6) and peak[1]<12),"H6":verdict(m["не кофе"]-m["кофе"]>=3)}
print(verdicts)

{'H1': 'подтвердилась', 'H2': 'не подтвердилась', 'H4': 'не подтвердилась', 'H5': 'подтвердилась', 'H6': 'не подтвердилась'}


**Вывод:** соединены три таблицы и проверены шесть гипотез. Рост сети и сетевой пик подтверждены; совпадение лидеров, пороги чека и маржи — нет. Рекомендуем изучить PRM-1 и спланировать отдельное измерение влияния выдачи карты; гипотезу сезонного напитка проверять экспериментом. Предсказание индивидуального чека по этим группам невозможно.